# WK/RK 20·21번 결과 다시 생성
DB 조회 없이 저장된 19번 전체 point 자료를 읽습니다. 기존 stages의 21번 결과를 덮어씁니다. 입력 19번 및 다른 단계 파일은 수정하지 않습니다. 04번 자료가 있으면 wafer 선정과 실측 목록도 비교합니다.


## 00. 경로와 저장 설정

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

BASE_DIR = Path.cwd()  # C:\PHOTO_ANALYSIS\BASIC에 놓고 실행
STAGE_DIR = BASE_DIR / "output_decorrect_notebook" / "stages"
SAVE_PARQUET, SAVE_EXCEL = True, True
PREVIEW_ROWS, EXCEL_PREVIEW_ROWS = 10, 10000

def read_full_stage(name):
    errors = []
    for suffix in [".parquet", ".xlsx"]:
        path = STAGE_DIR / (name + suffix)
        if path.is_file():
            try:
                result = pd.read_parquet(path) if suffix == ".parquet" else pd.read_excel(path)
                print(f"입력: {path.name} ({len(result):,} rows)")
                return result
            except Exception as exc:
                errors.append(f"{path.name}: {exc}")
    raise FileNotFoundError(f"{name}: 전체 Parquet/Excel이 필요합니다. _sample 파일은 사용하지 않습니다. " + " | ".join(errors))

def save_stage(name, frame, columns=None):
    STAGE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[{name}] {len(frame):,} rows × {len(frame.columns)} columns")
    selected = [col for col in columns if col in frame] if columns else list(frame.columns)
    display(frame[selected].head(PREVIEW_ROWS))
    if SAVE_PARQUET:
        path = STAGE_DIR / f"{name}.parquet"
        frame.to_parquet(path, index=False)
        display(FileLink(os.path.relpath(path, Path.cwd())))
    if SAVE_EXCEL:
        exported = frame.head(min(EXCEL_PREVIEW_ROWS, 1048575))
        suffix = f"_sample_{len(exported)}" if len(exported) < len(frame) else ""
        path = STAGE_DIR / f"{name}{suffix}.xlsx"
        exported.to_excel(path, index=False, engine="openpyxl")
        display(FileLink(os.path.relpath(path, Path.cwd())))


## 01. 저장된 실측 point 읽기

In [ ]:
df_adi_trocs = read_full_stage("19_ADI_TROCS_ATTACHED")
df_oco_trocs = read_full_stage("19_OCO_TROCS_ATTACHED")


## 18. 실제 존재하는 그룹만 집계

In [ ]:
WAFER_KEYS = ["LOT_ID", "Wafer"]
SHOT_KEYS = WAFER_KEYS + ["fcp_x", "fcp_y"]
POINT_KEYS = ["UNIQUE_ID", "LOT_ID", "Wafer", "GROUP", "TEST", "DieX", "DieY", "apc_hist_index_no", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]

def numeric(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df
def summarize_values(frame, keys, value_cols):
    columns = keys + ['n_points'] + [f'{col}_{stat}' for col in value_cols for stat in ['n_valid', 'mean', 'std', 'rms', 'max_abs']]
    if frame.empty:
        return pd.DataFrame(columns=columns)
    grouped = frame.groupby(keys, dropna=False, observed=True)
    result = grouped.size().rename('n_points').to_frame()
    for col in value_cols:
        work = frame[keys].copy()
        work['_value'] = pd.to_numeric(frame[col], errors='coerce')
        work['_square'], work['_abs'] = (work['_value'] ** 2, work['_value'].abs())
        stats = work.groupby(keys, dropna=False, observed=True).agg(n_valid=('_value', 'count'), mean=('_value', 'mean'), std=('_value', 'std'), mean_square=('_square', 'mean'), max_abs=('_abs', 'max'))
        stats['rms'] = np.sqrt(stats.pop('mean_square'))
        result = result.join(stats.add_prefix(col + '_'))
    return result.reset_index().reindex(columns=columns)


## 20. WK/RK 회귀와 shot index

In [ ]:
WK_RK_MIN_POINTS = 19

def osr_wk20p_rk20p(x, y, rx, ry):
    """
    회사 코드의 osr_wk20p_rk20p 구조를 그대로 사용.
    x, y   : wafer/field level 좌표 -> fcp_x, fcp_y
    rx, ry : shot 내부 좌표         -> coordinate_X, coordinate_Y
    반환:
        X_dx: X 방향 회귀용 design matrix (N x 19)
        X_dy: Y 방향 회귀용 design matrix (N x 19)
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(x)),
        x / 1e6, -y / 1e6,
        (x ** 2) / 1e12, (x * y) / 1e12, (y ** 2) / 1e12,
        (x ** 3) / 1e15, (x ** 2 * y) / 1e15, (x * y ** 2) / 1e15, (y ** 3) / 1e15,
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
    ]).T
    X_dy = np.vstack([
        np.ones(len(y)),
        y / 1e6, x / 1e6,
        (y ** 2) / 1e12, (y * x) / 1e12, (x ** 2) / 1e12,
        (y ** 3) / 1e15, (y ** 2 * x) / 1e15, (y * x ** 2) / 1e15, (x ** 3) / 1e15,
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12,
    ]).T
    return X_dx, X_dy
COEFF_KEYS_DX = [
    "WK1", "WK3", "WK5", "WK7", "WK9", "WK11", "WK13", "WK15", "WK17", "WK19",
    "RK3", "RK5", "RK7", "RK9", "RK11", "RK13", "RK15", "RK17", "RK19",
]
COEFF_KEYS_DY = [
    "WK2", "WK4", "WK6", "WK8", "WK10", "WK12", "WK14", "WK16", "WK18", "WK20",
    "RK4", "RK6", "RK8", "RK10", "RK12", "RK14", "RK16", "RK18", "RK20",
]
WK_COLS_DX = COEFF_KEYS_DX[:10]
RK_COLS_DX = COEFF_KEYS_DX[10:]
WK_COLS_DY = COEFF_KEYS_DY[:10]
RK_COLS_DY = COEFF_KEYS_DY[10:]

def normalize_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def fit_lstsq(X, y):
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ coef
    return coef, pred

def run_company_joint_regression(df, label="ADI", min_points=19):
    """
    wafer 단위로 raw_x/raw_y를 회사 룰 osr_wk20p_rk20p로 동시 회귀.
    입력:
      - raw_x, raw_y
      - fcp_x, fcp_y
      - coordinate_X, coordinate_Y
      - LOT_ID, Wafer
    출력:
      - wk_fit_x/y
      - rk_fit_x/y
      - total_fit_x/y
      - raw_minus_total_fit_x/y
      - wafer별 coefficient dataframe
    """
    df = df.copy()
    required_cols = [
        "LOT_ID", "Wafer",
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 필수 컬럼 누락: {missing}")
    normalize_numeric(df, [
        "raw_x", "raw_y",
        "fcp_x", "fcp_y",
        "coordinate_X", "coordinate_Y",
    ])
    result_cols = [
        "wk_fit_x", "wk_fit_y",
        "rk_fit_x", "rk_fit_y",
        "total_fit_x", "total_fit_y",
        "raw_minus_total_fit_x", "raw_minus_total_fit_y",
    ]
    for c in result_cols:
        df[c] = np.nan
    coef_rows = []
    summary_rows = []
    for (lot_id, wafer), grp in df.groupby(["LOT_ID", "Wafer"], dropna=False, observed=True):
        idx = grp.index
        valid_mask = (
            grp["raw_x"].notna() &
            grp["raw_y"].notna() &
            grp["fcp_x"].notna() &
            grp["fcp_y"].notna() &
            grp["coordinate_X"].notna() &
            grp["coordinate_Y"].notna()
        )
        work = grp.loc[valid_mask].copy()
        n_valid = len(work)
        if n_valid < min_points:
            summary_rows.append({
                "LOT_ID": lot_id,
                "Wafer": wafer,
                "n_total": len(grp),
                "n_valid": n_valid,
                "status": "skip_too_few_points",
            })
            continue
        x = work["fcp_x"].to_numpy(dtype=float)
        y = work["fcp_y"].to_numpy(dtype=float)
        rx = work["coordinate_X"].to_numpy(dtype=float)
        ry = work["coordinate_Y"].to_numpy(dtype=float)
        raw_x = work["raw_x"].to_numpy(dtype=float)
        raw_y = work["raw_y"].to_numpy(dtype=float)
        X_dx, X_dy = osr_wk20p_rk20p(x, y, rx, ry)
        coef_x, pred_x = fit_lstsq(X_dx, raw_x)
        coef_y, pred_y = fit_lstsq(X_dy, raw_y)
        wk_fit_x = X_dx[:, :10] @ coef_x[:10]
        rk_fit_x = X_dx[:, 10:] @ coef_x[10:]
        wk_fit_y = X_dy[:, :10] @ coef_y[:10]
        rk_fit_y = X_dy[:, 10:] @ coef_y[10:]
        total_fit_x = wk_fit_x + rk_fit_x
        total_fit_y = wk_fit_y + rk_fit_y
        residual_x = raw_x - total_fit_x
        residual_y = raw_y - total_fit_y
        valid_idx = work.index
        df.loc[valid_idx, "wk_fit_x"] = wk_fit_x
        df.loc[valid_idx, "rk_fit_x"] = rk_fit_x
        df.loc[valid_idx, "total_fit_x"] = total_fit_x
        df.loc[valid_idx, "raw_minus_total_fit_x"] = residual_x
        df.loc[valid_idx, "wk_fit_y"] = wk_fit_y
        df.loc[valid_idx, "rk_fit_y"] = rk_fit_y
        df.loc[valid_idx, "total_fit_y"] = total_fit_y
        df.loc[valid_idx, "raw_minus_total_fit_y"] = residual_y
        coef_row = {
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
        }
        for name, val in zip(COEFF_KEYS_DX, coef_x):
            coef_row[f"{name}_X"] = val
        for name, val in zip(COEFF_KEYS_DY, coef_y):
            coef_row[f"{name}_Y"] = val
        coef_rows.append(coef_row)
        summary_rows.append({
            "LOT_ID": lot_id,
            "Wafer": wafer,
            "n_total": len(grp),
            "n_valid": n_valid,
            "rms_raw_x": float(np.sqrt(np.nanmean(raw_x ** 2))),
            "rms_raw_y": float(np.sqrt(np.nanmean(raw_y ** 2))),
            "rms_residual_x": float(np.sqrt(np.nanmean(residual_x ** 2))),
            "rms_residual_y": float(np.sqrt(np.nanmean(residual_y ** 2))),
            "status": "ok",
        })
    coef_df = pd.DataFrame(coef_rows)
    summary_df = pd.DataFrame(summary_rows)
    print(f"[{label}] regression 완료")
    print(f"  - wafer 수: {df[['LOT_ID','Wafer']].drop_duplicates().shape[0]}")
    print(f"  - coefficient rows: {len(coef_df)}")
    print(f"  - summary rows: {len(summary_df)}")
    return df, coef_df, summary_df

def attach_measured_shot_indices(summary, points):
    """Attach the measured shot grid index; (0, 0) is the map-shift origin, not the sampling mean."""
    keys = ["LOT_ID", "Wafer", "fcp_x", "fcp_y"]
    work = points[keys].copy()
    for axis, die in [("x", "DieX"), ("y", "DieY")]:
        values = pd.to_numeric(points[die], errors="coerce").astype(float) if die in points else pd.Series(np.nan, index=points.index)
        pitch_col, shift_col = f"STEP_PITCH_{axis.upper()}", f"MAP_SHIFT_{axis.upper()}"
        if pitch_col in points and shift_col in points:
            pitch = pd.to_numeric(points[pitch_col], errors="coerce").astype(float)
            shift = pd.to_numeric(points[shift_col], errors="coerce").astype(float)
            center = pd.to_numeric(points[f"fcp_{axis}"], errors="coerce").astype(float)
            derived = ((center - shift) / pitch.where(pitch.gt(0))).replace([np.inf, -np.inf], np.nan)
            usable = derived.notna() & np.isclose(derived, derived.round(), rtol=0, atol=1e-5)
            conflict = values.notna() & usable & ~np.isclose(values, derived.round(), rtol=0, atol=1e-5)
            if conflict.any():
                raise ValueError(f"{die}: 실측 index와 (fcp - MAP_SHIFT) / STEP_PITCH가 다릅니다. 좌표 메타데이터를 확인하세요.")
            values = values.fillna(derived.round().where(usable))
        invalid = values.notna() & (~np.isfinite(values) | ~np.isclose(values, values.round(), rtol=0, atol=1e-5))
        if invalid.any():
            raise ValueError(f"{die}: 정수가 아닌 shot index가 있습니다. 임의로 재번호를 부여하지 않습니다.")
        work[die] = values.round().astype("Int64")
    groups = work.groupby(keys, observed=True, dropna=False, sort=False)
    conflicts = groups[["DieX", "DieY"]].nunique(dropna=True).gt(1).any(axis=1)
    if conflicts.any():
        raise ValueError("같은 LOT/wafer/shot 중심에 여러 DieX/DieY가 있습니다. 노광 이력 또는 좌표를 확인하세요.")
    mapping = groups[["DieX", "DieY"]].first().reset_index()
    result = summary.merge(mapping, on=keys, how="left", validate="1:1", indicator="_shot_match")
    if result["_shot_match"].ne("both").any() or len(result) != len(mapping):
        raise ValueError("Shot 통계의 key가 실측 point의 shot 목록과 다릅니다. 18번 summarize_values를 다시 실행하세요.")
    result = result.drop(columns="_shot_match")
    result["shot_index_status"] = np.where(result[["DieX", "DieY"]].notna().all(axis=1), "ok", "missing_geometry")
    if result["shot_index_status"].ne("ok").any():
        print("일부 shot의 DieX/DieY를 복원할 수 없습니다. 08/19 단계의 index·pitch·map shift를 확인하세요.")
    first = ["LOT_ID", "Wafer", "DieX", "DieY", "fcp_x", "fcp_y", "shot_index_status"]
    return result[first + [col for col in summary.columns if col not in first]]


## 21. 재계산 및 저장

In [ ]:
def split_wk_rk_coefficients(coeff):
    base = ["LOT_ID", "Wafer", "n_total", "n_valid"]
    wk = base + [f"{key}_{axis}" for axis, names in [("X", WK_COLS_DX), ("Y", WK_COLS_DY)] for key in names]
    rk = base + [f"{key}_{axis}" for axis, names in [("X", RK_COLS_DX), ("Y", RK_COLS_DY)] for key in names]
    return coeff.reindex(columns=wk), coeff.reindex(columns=rk)

def wk_rk_rank_diagnostics(frame, summary):
    rank_columns = WAFER_KEYS + ["rank_x", "rank_y", "n_parameters_per_axis", "rank_deficient"]
    if frame.empty:
        return summary.reindex(columns=list(dict.fromkeys(list(summary.columns) + rank_columns)))
    rows = []
    cols = ["raw_x", "raw_y", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]
    for key, grp in frame.groupby(WAFER_KEYS, dropna=False, observed=True):
        valid = numeric(grp[cols].copy(), cols).dropna()
        if not valid.empty:
            mx, my = osr_wk20p_rk20p(*(valid[col].to_numpy(dtype=float) for col in cols[2:]))
            rank_x, rank_y = np.linalg.matrix_rank(mx), np.linalg.matrix_rank(my)
        else:
            rank_x, rank_y = 0, 0
        rows.append({"LOT_ID": key[0], "Wafer": key[1], "rank_x": rank_x, "rank_y": rank_y, "n_parameters_per_axis": 19, "rank_deficient": rank_x < 19 or rank_y < 19})
    rank = pd.DataFrame(rows, columns=rank_columns)
    return summary.reindex(columns=list(dict.fromkeys(WAFER_KEYS + list(summary.columns)))).merge(rank, on=WAFER_KEYS, how="left", validate="1:1")

df_adi_reg, coef_adi, summary_adi = run_company_joint_regression(df_adi_trocs, label="ADI", min_points=WK_RK_MIN_POINTS)
df_oco_reg, coef_oco, summary_oco = run_company_joint_regression(df_oco_trocs, label="OCO", min_points=WK_RK_MIN_POINTS)
df_wk_coeff_adi, df_rk_coeff_adi = split_wk_rk_coefficients(coef_adi)
df_wk_coeff_oco, df_rk_coeff_oco = split_wk_rk_coefficients(coef_oco)
df_wk_rk_diag_adi, df_wk_rk_diag_oco = wk_rk_rank_diagnostics(df_adi_reg, summary_adi), wk_rk_rank_diagnostics(df_oco_reg, summary_oco)
wk_rk_values = ["raw_x", "raw_y", "wk_fit_x", "wk_fit_y", "rk_fit_x", "rk_fit_y", "raw_minus_total_fit_x", "raw_minus_total_fit_y"]
df_wk_rk_wafer_adi, df_wk_rk_wafer_oco = summarize_values(df_adi_reg, WAFER_KEYS, wk_rk_values), summarize_values(df_oco_reg, WAFER_KEYS, wk_rk_values)
df_wk_rk_shot_adi, df_wk_rk_shot_oco = attach_measured_shot_indices(summarize_values(df_adi_reg, SHOT_KEYS, wk_rk_values), df_adi_reg), attach_measured_shot_indices(summarize_values(df_oco_reg, SHOT_KEYS, wk_rk_values), df_oco_reg)
df_wk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_adi_reg]].copy()
df_wk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["wk_fit_x", "wk_fit_y"] if col in df_oco_reg]].copy()
df_rk_removed_adi = df_adi_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_adi_reg]].copy()
df_rk_removed_oco = df_oco_reg[[col for col in POINT_KEYS + ["rk_fit_x", "rk_fit_y"] if col in df_oco_reg]].copy()
wk_rk_results = [
    ("ADI", df_adi_reg, df_wk_coeff_adi, df_rk_coeff_adi, df_wk_removed_adi, df_rk_removed_adi, df_wk_rk_diag_adi, df_wk_rk_wafer_adi, df_wk_rk_shot_adi),
    ("OCO", df_oco_reg, df_wk_coeff_oco, df_rk_coeff_oco, df_wk_removed_oco, df_rk_removed_oco, df_wk_rk_diag_oco, df_wk_rk_wafer_oco, df_wk_rk_shot_oco),
]
for mode, reg, wk, rk, wk_points, rk_points, diag, wafer, shot in wk_rk_results:
    save_stage(f"21_{mode}_WK_RK_REMOVED", reg, columns=POINT_KEYS + wk_rk_values)
    save_stage(f"21_{mode}_WK_WAFER_COEFFICIENTS", wk)
    save_stage(f"21_{mode}_RK_WAFER_COEFFICIENTS", rk)
    save_stage(f"21_{mode}_WK_REMOVED_POINTS", wk_points)
    save_stage(f"21_{mode}_RK_REMOVED_POINTS", rk_points)
    save_stage(f"21_{mode}_WK_RK_WAFER_DIAGNOSTICS", diag)
    save_stage(f"21_{mode}_WK_RK_WAFER_DATA", wafer)
    save_stage(f"21_{mode}_WK_RK_SHOT_DATA", shot)


## 22. 실측 목록과 LOT 선정 검증

In [ ]:
def wafer_counts(frame):
    return frame.groupby(WAFER_KEYS, observed=True, dropna=False).size().rename("n_measured_points").reset_index()

for mode, source, wafer, shot in [("ADI", df_adi_trocs, df_wk_rk_wafer_adi, df_wk_rk_shot_adi), ("OCO", df_oco_trocs, df_wk_rk_wafer_oco, df_wk_rk_shot_oco)]:
    measured = wafer_counts(source)
    check = measured.merge(wafer[WAFER_KEYS + ["n_points"]], on=WAFER_KEYS, how="outer", validate="1:1", indicator=True)
    assert check["_merge"].eq("both").all() and check["n_measured_points"].eq(check["n_points"]).all()
    assert shot["n_points"].gt(0).all() and shot["n_points"].sum() == len(source)
    selected_name = f"04_LOTINFO_{mode}_SELECTED"
    if any((STAGE_DIR / (selected_name + ext)).is_file() for ext in [".parquet", ".xlsx"]):
        selected = read_full_stage(selected_name).rename(columns={"lotid": "LOT_ID", "slotid": "Wafer"})
        selected = selected[WAFER_KEYS].drop_duplicates()
        scope = selected.merge(measured, on=WAFER_KEYS, how="outer", validate="1:1", indicator=True)
        scope["scope_status"] = scope.pop("_merge").astype(str).map({"both": "selected_and_measured", "left_only": "selected_without_stage19_points", "right_only": "stage19_not_in_selected"})
        save_stage(f"21_{mode}_WK_RK_WAFER_SCOPE", scope)
        if scope["scope_status"].eq("stage19_not_in_selected").any():
            print(f"{mode}: 04와 19의 실행 대상이 다릅니다. 저장 파일이 같은 실행 자료인지 확인하세요.")
    display(wafer[WAFER_KEYS + ["n_points"]])
    print(f"{mode}: 실측 wafer {len(wafer)}, 실측 shot {len(shot)} — 빈 조합 없음")
